# Encrypted RAG Chatbot with CyborgDB + NVIDIA NIM

> **Last validated: 2026-06-04 | CyborgDB v0.17 (disk-backed)**

CyborgDB 0.17 runs as a single-node service with an on-disk encrypted index — no external PostgreSQL or Redis cluster needed. Combined with a local NIM LLM, the entire pipeline (embeddings, vector store, and inference) stays on-prem.

## 🚀 Quick Start Guide

This notebook uses a **local NVIDIA NIM LLM** instead of OpenAI for completely private, on-premises deployment!

---

### Step 1: Start Your NIM LLM

Before running this notebook, make sure you have a NIM LLM running locally. For example:

```bash
# Example: Running Llama-3.1-8B-Instruct with NIM
docker run -d --gpus all \
  -p 8001:8000 \
  --name nim-llm \
  nvcr.io/nvidia/nim/meta/llama-3.1-8b-instruct:latest
```

In the configuration cell below, you'll set:
- `NIM_BASE_URL`: The URL where your NIM LLM is running (default: `http://localhost:8001/v1`)
- `NIM_MODEL_NAME`: The model identifier (e.g., `meta/llama-3.1-8b-instruct`)

---

### Step 2: Run All Cells

Simply click **Runtime → Run all** and the notebook will:
- ✅ Install all Python dependencies
- ✅ After install, **manually click "Runtime → Restart and run all"** to continue
- ✅ Start the CyborgDB service
- ✅ Launch the Gradio chatbot interface

**Note:** The first run will automatically restart the runtime after installing packages. This prevents version conflicts. Just run all cells again after the restart!

---

### Step 3: Open the Public URL

When the last cell finishes running, you'll see a **local Gradio URL** (e.g., `http://127.0.0.1:7860`).

**Click the URL** to open the chatbot interface, then:
1. Upload PDF or TXT documents
2. Click "Create Encrypted Vector Store"
3. Start asking questions about your documents!

All vector embeddings are encrypted end-to-end, and your LLM stays completely local! 🔒

In [ ]:
# 0. Install dependencies and handle automatic restart if needed

import subprocess, sys, os

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

def restart():
    if IN_COLAB:
        print("🔄 Restarting Colab runtime...")
        print("Please click on \"Runtime > Restart and run all\" if it does not restart automatically.")
        
        # Wait for a moment to let the print flush
        import time
        time.sleep(2)

    else:
        from IPython import get_ipython
        print("🔄 Restarting Jupyter kernel...")
        print("Please re-run all notebook cells if it does not restart automatically.")
        get_ipython().kernel.do_shutdown(restart=True)
    
    raise SystemExit(0)

# TODO(v0.17): bump cyborgdb / cyborgdb-service pins to ==0.17.0 once published.
reqs = [
    "numpy","cyborgdb[langchain]==0.15.0","getpass4",
    "cyborgdb-service==0.15.0","scipy==1.17.1","scikit-learn==1.8.0",
    "transformers==5.3.0", "sentence-transformers==5.2.3",
    "langchain==0.3.26","langchain-community==0.3.27",
    "langchain-huggingface==0.3.1","langchain-openai==0.3.28",
    "pypdf==5.8.0","gradio==5.38.0",
    "cryptography==46.0.5"
]

# Check if any dependencies need to be installed or updated
print("📦 Checking dependencies...")
probe = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--dry-run", *reqs],
    capture_output=True, text=True
)
out = probe.stdout + probe.stderr

# If pip indicates it would install or update anything, do so and restart
if "Would install" in out or "Installing collected" in out:
    print("⬇️ Dependencies missing or outdated — installing and restarting...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-U", *reqs], check=True)
    restart()
else:
    print("✅ All requirements already satisfied — no restart needed.")

In [ ]:
# 1. Configure CyborgDB database type and NIM settings

# Single-node disk-backed encrypted index (RocksDB under the hood).
# Service env var still uses the legacy 'standalone'
CYBORGDB_DB_TYPE = 'standalone'

# Configure NIM LLM settings
NIM_BASE_URL = 'http://localhost:8001/v1'  # Update to your NIM endpoint
NIM_MODEL_NAME = 'openai/gpt-oss-20b'  # Update to match your NIM model

In [ ]:
# 2. Configure CyborgDB API key and environment variables

import os
from cyborgdb import get_demo_api_key

# Configure API key or demo API key (expires in 1 hour)
CYBORGDB_API_KEY = os.environ.get("CYBORGDB_API_KEY") or get_demo_api_key()
os.environ["CYBORGDB_API_KEY"] = CYBORGDB_API_KEY

# Set environment variables
os.environ["CYBORGDB_DB_TYPE"] = CYBORGDB_DB_TYPE
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["GRADIO_ANALYTICS_ENABLED"] = "false"

print(f"✅ Configuration complete:")
print(f"   Database: {CYBORGDB_DB_TYPE}")
print(f"   NIM URL: {NIM_BASE_URL}")
print(f"   NIM Model: {NIM_MODEL_NAME}")

In [ ]:
# 2a. Start NIM Docker Container (if not already running)

import subprocess
import time
import requests

# Configuration
NIM_CONTAINER_NAME = "nim-llm"
NIM_IMAGE = "nvcr.io/nim/openai/gpt-oss-20b:latest"
NIM_HOST_PORT = 8001
NIM_CONTAINER_PORT = 8000
LOCAL_NIM_CACHE = os.path.expanduser("~/.cache/nim")

def run_cmd(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True, check=False)

# Create cache directory
os.makedirs(LOCAL_NIM_CACHE, exist_ok=True)
print(f"📁 NIM cache directory: {LOCAL_NIM_CACHE}")

# Check if container already exists and is running
check_container = run_cmd(f"docker ps -a --filter name={NIM_CONTAINER_NAME} --format '{{{{.Status}}}}'")
container_status = check_container.stdout.strip()

if "Up" in container_status:
    print(f"✅ NIM container '{NIM_CONTAINER_NAME}' is already running")
elif container_status:
    print(f"🔄 NIM container '{NIM_CONTAINER_NAME}' exists but is not running. Starting it...")
    start_result = run_cmd(f"docker start {NIM_CONTAINER_NAME}")
    if start_result.returncode != 0:
        print(f"❌ Failed to start container: {start_result.stderr}")
        raise RuntimeError("Failed to start NIM container")
else:
    print(f"🚀 Launching new NIM container '{NIM_CONTAINER_NAME}'...")
    print(f"   Image: {NIM_IMAGE}")
    print(f"   Port: {NIM_HOST_PORT} -> {NIM_CONTAINER_PORT}")
    print("   This may take several minutes on first run (downloading model)...")
    
    docker_cmd = f"""docker run -d \\
        --gpus all \\
        --shm-size=16GB \\
        -e NGC_API_KEY \\
        -v "{LOCAL_NIM_CACHE}:/opt/nim/.cache" \\
        -p {NIM_HOST_PORT}:{NIM_CONTAINER_PORT} \\
        --name {NIM_CONTAINER_NAME} \\
        {NIM_IMAGE}"""
    
    result = run_cmd(docker_cmd)
    
    if result.returncode != 0:
        print(f"❌ Failed to launch container: {result.stderr}")
        raise RuntimeError("Failed to launch NIM container")
    
    print(f"✅ Container '{NIM_CONTAINER_NAME}' launched successfully")

# Wait for NIM to be ready
print("⏳ Waiting for NIM service to be ready...")
nim_url = f"http://localhost:{NIM_HOST_PORT}/v1/health"

for i in range(120):  # Wait up to 10 minutes
    try:
        response = requests.get(nim_url, timeout=2)
        if response.status_code == 200:
            print(f"\n✅ NIM service is ready at http://localhost:{NIM_HOST_PORT}")
            break
    except:
        pass
    
    if i % 10 == 0 and i > 0:
        print(f"   Still waiting... ({i}/120 - this can take several minutes on first run)")
    time.sleep(5)
else:
    print("\n⚠️ Timeout waiting for NIM service. Checking logs...")
    logs = run_cmd(f"docker logs --tail 50 {NIM_CONTAINER_NAME}")
    print(logs.stdout)
    print(logs.stderr)
    raise RuntimeError("NIM service failed to start in time")

# Show container info
print(f"\n📊 NIM Container Status:")
run_cmd(f"docker ps --filter name={NIM_CONTAINER_NAME}")
print(f"   View logs: docker logs -f {NIM_CONTAINER_NAME}")
print(f"   Stop: docker stop {NIM_CONTAINER_NAME}")
print(f"   Remove: docker rm {NIM_CONTAINER_NAME}")

In [ ]:
# 2b. Configure NGC API Key for NIM

import os
import getpass

# Get NGC API key from environment or prompt
NGC_API_KEY = os.environ.get("NGC_API_KEY")
if not NGC_API_KEY:
    print("NGC API Key not found in environment.")
    print("Get your free NGC API key from: https://catalog.ngc.nvidia.com/")
    print("Sign in → Profile → Setup → Generate API Key")
    NGC_API_KEY = getpass.getpass("Enter your NGC API Key: ")

# Set as environment variable for Docker
os.environ["NGC_API_KEY"] = NGC_API_KEY

print("✅ NGC API Key configured")

In [ ]:
# 3. Start CyborgDB service and wait for it to be ready

import subprocess
import time
import requests

# Start service
print("🔄 Starting CyborgDB service...")
log_file = "cyborgdb_service.log"

process = subprocess.Popen(
    "cyborgdb-service",
    stdout=open(log_file, "w"),
    stderr=subprocess.STDOUT,
    env=dict(os.environ)
)

# Wait for startup
for i in range(60):
    try:
        response = requests.get("http://localhost:8000/v1/health", timeout=1)
        if response.status_code == 200:
            print("\n✅ CyborgDB service started!")
            break
    except:
        pass
    time.sleep(1)
    if i % 5 == 0 and i > 0:
        print(f"   Waiting... ({i}/60)")
else:
    print("\n❌ Service failed to start. Logs:")
    with open(log_file, 'r') as f:
        print(f.read()[-1000:])
    raise RuntimeError("Failed to start service")

In [ ]:
# 4. Main Application Code

import os
import pathlib
import base64
import uuid
import numpy as np
import langchain.chains
import langchain.chains.combine_documents
import langchain.docstore.document
import langchain.document_loaders
import langchain.prompts
import langchain.text_splitter
import langchain_community.chat_message_histories
import langchain_core.runnables.history
import langchain_huggingface
import langchain_openai
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
import cyborgdb
from cyborgdb.integrations.langchain import CyborgVectorStore
import gradio as gr

print("✅ Libraries imported")

# Settings
EMBEDDING_MODEL_ID = "all-MiniLM-L6-v2"
CHUNK_SIZE = 2048
CHUNK_OVERLAP = 128
CYBORGDB_HOST = "http://localhost:8000"
CYBORGDB_KEYS = {}
VECTORSTORE_STORE = {}
CHAT_HISTORY_STORE = {}
CHAINS_STORE = {}
LAST_QUERY_INFO = {}
MOST_RECENT_PROMPT = None

print("✅ Settings configured")

# Functions
def split_documents(file_path: str) -> list:
    file = pathlib.Path(file_path)
    if not file.exists():
        raise ValueError("File not found")
    if file.suffix == ".pdf":
        loader = langchain.document_loaders.PyPDFLoader(str(file))
        pages = loader.load()
        splitter = langchain.text_splitter.RecursiveCharacterTextSplitter(
            chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP
        )
        return splitter.split_documents(pages)
    else:
        loader = langchain.document_loaders.TextLoader(str(file))
        pages = loader.load()
        splitter = langchain.text_splitter.RecursiveCharacterTextSplitter(
            chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP
        )
        return splitter.split_documents(pages)

def generate_encryption_key() -> bytes:
    """Generate a 256-bit AES-GCM key."""
    return AESGCM.generate_key(bit_length=256)

def encrypt_chunk(text: str, key: bytes) -> str:
    aesgcm = AESGCM(key)
    nonce = os.urandom(12)
    ciphertext = aesgcm.encrypt(nonce, text.encode(), None)
    return base64.b64encode(nonce + ciphertext).decode()

def load_vectorstore(session_id: str) -> CyborgVectorStore:
    if session_id in VECTORSTORE_STORE:
        return VECTORSTORE_STORE[session_id]
    if session_id not in CYBORGDB_KEYS:
        # Generate key using cryptography library directly
        CYBORGDB_KEYS[session_id] = generate_encryption_key()
    vector_store = CyborgVectorStore(
        index_name=f"session_{session_id}",
        index_key=CYBORGDB_KEYS[session_id],
        api_key=CYBORGDB_API_KEY,
        base_url=CYBORGDB_HOST,
        embedding=EMBEDDING_MODEL_ID,
        index_type="ivfflat",
        metric="cosine",
    )
    VECTORSTORE_STORE[session_id] = vector_store
    return vector_store

def upload_documents(file_paths: list[str], session_id: str) -> str:
    try:
        if not file_paths:
            return "❌ No files selected"

        print(f"Processing {len(file_paths)} files for session {session_id}")
        vectorstore = load_vectorstore(session_id)
        encryption_key = bytes(CYBORGDB_KEYS[session_id])
        all_docs = []
        previews = []

        for file_path in file_paths:
            print(f"Processing: {file_path}")
            docs = split_documents(file_path)
            all_docs.extend(docs)
            for i, doc in enumerate(docs[:3]):
                encrypted = encrypt_chunk(doc.page_content, encryption_key)
                previews.append(f"  Chunk {i+1}: {encrypted[:60]}...")

        if all_docs:
            print(f"Adding {len(all_docs)} documents to vector store")
            vectorstore.add_documents(all_docs)
            print("Training index...")
            vectorstore.index.train()
            print("Index trained successfully")

        result = [
            f"✅ Processed {len(file_paths)} file(s) with {len(all_docs)} chunks",
            "",
            "🔐 Encrypted chunk previews:"
        ]
        result.extend(previews[:5])
        if len(all_docs) > 5:
            result.append(f"  ... and {len(all_docs) - 5} more encrypted chunks")
        return "\n".join(result)
    except Exception as e:
        import traceback
        error_msg = traceback.format_exc()
        print(f"ERROR in upload_documents: {error_msg}")
        return f"❌ Error: {str(e)}\n\nSee logs for details."

def load_chain(session_id: str, system_prompt: str):
    print(f"Loading chain for session {session_id}")
    # Use NIM endpoint with OpenAI-compatible API
    llm = langchain_openai.ChatOpenAI(
        base_url=NIM_BASE_URL,
        model=NIM_MODEL_NAME,
        api_key="not-used",  # NIM doesn't require API key
        max_tokens=500,
        temperature=0.7
    )
    vectordb = load_vectorstore(session_id)
    retriever = vectordb.as_retriever(search_kwargs={"k": 3})

    contextualize_prompt = langchain.prompts.ChatPromptTemplate.from_messages([
        ("system", "Given chat history and a new question, reformulate the question to be standalone. If it's already standalone, return as is."),
        langchain.prompts.MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ])

    history_aware_retriever = langchain.chains.create_history_aware_retriever(
        llm, retriever, contextualize_prompt
    )

    qa_prompt = langchain.prompts.ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        langchain.prompts.MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ])

    document_chain = langchain.chains.combine_documents.create_stuff_documents_chain(llm, qa_prompt)
    rag_chain = langchain.chains.create_retrieval_chain(history_aware_retriever, document_chain)

    return langchain_core.runnables.history.RunnableWithMessageHistory(
        rag_chain,
        get_session_history=lambda: CHAT_HISTORY_STORE.get(
            session_id,
            langchain_community.chat_message_histories.ChatMessageHistory()
        ),
        input_messages_key="input",
        history_messages_key="chat_history",
        output_messages_key="answer",
    )

def create_comparison_html(query: str, result_docs=None, plaintext_embeddings=None, encrypted_embeddings=None):
    """Create HTML showing plaintext vs encrypted vectors."""
    if not result_docs or not plaintext_embeddings:
        return f"""
        <div style="margin: 20px 0; border: 2px solid #e2e8f0; border-radius: 12px; overflow: hidden; background: white; box-shadow: 0 4px 6px rgba(0,0,0,0.1);">
            <div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 20px; text-align: center;">
                <h2 style="color: white; margin: 0; font-size: 24px;">🔒 Retrieved Results: Plaintext vs Encrypted Vectors</h2>
            </div>
            <div style="padding: 40px; text-align: center; color: #666;">
                <p style="font-size: 16px;">Query: "<strong>{query}</strong>"</p>
                <p style="margin-top: 16px; color: #e53e3e; font-weight: 600;">⚠️ No documents found. Please upload documents first!</p>
            </div>
        </div>
        """

    results_html = ""
    for i, (doc, plaintext_emb, encrypted_emb) in enumerate(zip(result_docs, plaintext_embeddings, encrypted_embeddings)):
        content_preview = doc[:150] if len(doc) > 150 else doc
        plaintext_preview = "[" + ", ".join([f"{x:.4f}" for x in plaintext_emb[:20]]) + ", ...]"
        encrypted_preview = encrypted_emb[:150] + "..." if len(encrypted_emb) > 150 else encrypted_emb

        results_html += f"""
        <div style="margin-bottom: 16px; padding: 14px; background: #fafafa; border-radius: 8px; border: 1px solid #e2e8f0;">
            <div style="font-weight: 600; color: #333; margin-bottom: 10px; font-size: 15px;">
                📄 Result #{i+1}
            </div>
            <div style="margin-bottom: 12px; padding: 10px; background: white; border-radius: 6px; border: 1px solid #e2e8f0;">
                <div style="font-size: 14px; color: #666; font-weight: 600; margin-bottom: 4px;">
                    📝 Document Content:
                </div>
                <div style="font-size: 14px; color: #2d3748; line-height: 1.5;">
                    "{content_preview}..."
                </div>
            </div>
            <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 10px;">
                <div style="padding: 10px; background: #fff5f5; border: 2px solid #fc8181; border-radius: 6px;">
                    <div style="font-size: 14px; color: #c53030; font-weight: 600; margin-bottom: 6px;">
                        🔓 Plaintext Vector (first 20 dims):
                    </div>
                    <div style="background: white; padding: 6px; border-radius: 4px; font-family: 'Courier New', monospace; font-size: 14px; color: #4a5568; max-height: 120px; overflow-y: auto; line-height: 1.6; word-break: break-all;">
                        {plaintext_preview}
                    </div>
                </div>
                <div style="padding: 10px; background: #f0fff4; border: 2px solid #9ae6b4; border-radius: 6px;">
                    <div style="font-size: 14px; color: #22543d; font-weight: 600; margin-bottom: 6px;">
                        🔒 Encrypted Vector (first 150 chars):
                    </div>
                    <div style="background: white; padding: 6px; border-radius: 4px; font-family: 'Courier New', monospace; font-size: 14px; color: #4a5568; max-height: 120px; overflow-y: auto; word-break: break-all; line-height: 1.6;">
                        {encrypted_preview}
                    </div>
                </div>
            </div>
        </div>
        """

    return f"""
    <div style="--body-text-color: #1a202c; margin: 20px 0; border: 2px solid #e2e8f0; border-radius: 12px; overflow: hidden; background: white; 
  box-shadow: 0 4px 6px rgba(0,0,0,0.1);">
        <div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 20px; text-align: center;">
            <h2 style="color: white; margin: 0; font-size: 24px;">🔒 Retrieved Results: Plaintext vs Encrypted Vectors</h2>
        </div>
        <div style="padding: 16px; background: #f9fafb;">
            <div style="font-size: 15px; color: #666; margin-bottom: 16px; padding: 8px; background: white; border-radius: 6px;">
                <strong>Your Query:</strong> "{query}"
            </div>
            <div style="margin-bottom: 16px; padding: 16px; background: #edf2f7; border-radius: 8px;">
                <div style="text-align: center; margin-bottom: 12px;">
                    <p style="margin: 0; color: #2d3748; font-size: 16px; font-weight: 600;">
                        💡 Why Encrypt Embeddings?
                    </p>
                </div>
                <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 16px;">
                    <div style="background: white; padding: 12px; border-radius: 6px; border-left: 4px solid #fc8181;">
                        <div style="font-size: 14px; color: #c53030; font-weight: 600; margin-bottom: 8px;">
                            🚨 The Problem
                        </div>
                        <ul style="margin: 0; padding-left: 20px; font-size: 14px; color: #1a202c; line-height: 1.8;">
                            <li>Embeddings capture semantic meaning of your documents</li>
                            <li>Machine learning attacks can reconstruct original text</li>
                            <li>Database breaches expose sensitive information</li>
                            <li>Compliance regulations require data protection</li>
                        </ul>
                    </div>
                    <div style="background: white; padding: 12px; border-radius: 6px; border-left: 4px solid #9ae6b4;">
                        <div style="font-size: 14px; color: #22543d; font-weight: 600; margin-bottom: 8px;">
                            ✅ The Solution
                        </div>
                        <ul style="margin: 0; padding-left: 20px; font-size: 14px; color: #1a202c; line-height: 1.8;">
                            <li>CyborgDB encrypts vectors before storing them</li>
                            <li>Homomorphic encryption enables search on encrypted data</li>
                            <li>Breaches only expose encrypted, meaningless data</li>
                            <li>Maintains full RAG functionality with zero trust</li>
                        </ul>
                    </div>
                </div>
            </div>
            {results_html}
        </div>
    </div>
    """

def chat(message: str, history, session: str, system_prompt: str) -> tuple[str, str]:
    global MOST_RECENT_PROMPT, LAST_QUERY_INFO
    MOST_RECENT_PROMPT = message

    try:
        print(f"\n=== Chat Query ===")
        print(f"Session: {session}")
        print(f"Message: {message}")

        if session not in CHAINS_STORE:
            if session not in CHAT_HISTORY_STORE:
                CHAT_HISTORY_STORE[session] = langchain_community.chat_message_histories.ChatMessageHistory()
            CHAINS_STORE[session] = load_chain(session, system_prompt)

        chain = CHAINS_STORE[session]
        vectorstore = load_vectorstore(session)
        encryption_key = bytes(CYBORGDB_KEYS[session])

        # Retrieve documents
        print("Retrieving documents...")
        retrieved_docs = vectorstore.similarity_search(message, k=3)
        print(f"Retrieved {len(retrieved_docs)} documents")

        # Process retrieved documents
        result_contents = []
        plaintext_embeddings = []
        encrypted_embeddings = []

        for i, doc in enumerate(retrieved_docs):
            content = doc.page_content
            result_contents.append(content)

            # Get plaintext embedding
            doc_plaintext_embedding = vectorstore.get_embeddings(content)
            plaintext_embeddings.append(doc_plaintext_embedding.tolist())

            # Create encrypted version
            doc_encrypted_embedding = encrypt_chunk(str(doc_plaintext_embedding.tolist()), encryption_key)
            encrypted_embeddings.append(doc_encrypted_embedding)

        # Store for display
        LAST_QUERY_INFO = {
            'query': message,
            'result_contents': result_contents,
            'plaintext_embeddings': plaintext_embeddings,
            'encrypted_embeddings': encrypted_embeddings
        }

        # Get answer
        print("Invoking chain...")
        response = chain.invoke(
            {"input": message},
            config={"configurable": {"session_id": session}}
        )

        answer = response.get("answer", "No answer returned")
        print(f"Answer: {answer[:200]}...")

        # Create comparison HTML
        comparison_html = create_comparison_html(
            message,
            result_contents,
            plaintext_embeddings,
            encrypted_embeddings
        )

        return answer, comparison_html

    except Exception as e:
        import traceback
        error_msg = traceback.format_exc()
        print(f"ERROR in chat: {error_msg}")
        error_html = create_comparison_html(message, None, None, None)
        return f"❌ Error: {str(e)}", error_html

print("✅ Functions defined")

# Gradio UI
def get_session(request: gr.Request) -> str:
    return request.session_hash

with gr.Blocks(title="🔒 Encrypted RAG Chatbot (NIM)") as demo:
    gr.Markdown("# 🔒 Encrypted RAG Chatbot with CyborgDB + NVIDIA NIM")
    gr.Markdown("Upload documents and ask questions using your local NIM LLM - see plaintext vs encrypted vectors!")

    session = gr.State(value=str(uuid.uuid4()))

    with gr.Row():
        with gr.Column(scale=2, variant="panel"):
            gr.Markdown("## 💬 Chat with Your Documents")
            system_prompt = gr.Textbox(
                label="System instruction",
                lines=2,
                value="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Keep the answer concise. {context}",
                visible=False
            )

            chatbot = gr.Chatbot(height=400, label="Conversation", type="tuples")

            with gr.Row():
                msg = gr.Textbox(
                    label="Your question",
                    placeholder="Ask a question about your documents...",
                    scale=4,
                    show_label=False
                )
                submit_btn = gr.Button("Send", variant="primary", scale=1)

            clear_btn = gr.Button("Clear Chat", size="sm")

        with gr.Column(scale=1, variant="panel"):
            gr.Markdown("## 📄 Upload Documents")
            file_input = gr.File(
                type="filepath",
                file_count="multiple",
                label="Select PDF or TXT files"
            )
            upload_btn = gr.Button(
                "🔒 Create Encrypted Vector Store",
                variant="primary",
                size="lg"
            )
            upload_output = gr.Textbox(
                show_label=False,
                lines=6,
                placeholder="Upload files and click the button above..."
            )

            upload_btn.click(
                fn=lambda files, sess: upload_documents(files if files else [], sess),
                inputs=[file_input, session],
                outputs=[upload_output]
            )

            gr.Markdown("---")
            gr.Markdown("""
            ### 📋 Quick Start
            1. Upload documents
            2. Create encrypted vector store
            3. Ask questions
            4. See vector comparison below
            
            **Using Local NIM LLM!**
            """)

    gr.Markdown("---")
    gr.Markdown("## 🔍 Retrieved Results: Vector Comparison")
    gr.Markdown("*Compare plaintext vs encrypted vectors from search results*")

    comparison_display = gr.HTML(
        create_comparison_html("Your query will appear here", None, None, None),
        label="Vector Comparison"
    )

    # Wire up chat
    def respond(message, history, sess, sys_prompt):
        answer, updated_comparison = chat(message, history, sess, sys_prompt)
        history = history + [[message, answer]]
        return "", history, updated_comparison

    msg.submit(
        respond,
        inputs=[msg, chatbot, session, system_prompt],
        outputs=[msg, chatbot, comparison_display]
    )

    submit_btn.click(
        respond,
        inputs=[msg, chatbot, session, system_prompt],
        outputs=[msg, chatbot, comparison_display]
    )

    clear_btn.click(lambda: [], None, chatbot)
    demo.load(get_session, None, session)

print("✅ Gradio interface created")

In [ ]:
# 5. Launch Gradio App

print("\n🚀 Launching Gradio app...")
demo.launch(share=False, debug=True, inline=False)
print("\n🎉 Chatbot is running! Click the local URL above.")